# Pratica 03 — Preprocessing linguistico

**Corso:** NLP Classico con Python · **Lezione 03** · **Tempo stimato:** 60-75 minuti
**Notebook di lezione:** `notebooks_clase/clase03_preprocesamiento_linguistico_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- I testi degli esempi sono nella cella di dati che segue il setup; sono brevi e scritti per il corso (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. La prima cella installa solo ciò che manca (`nltk`, `spacy` e i modelli di spaCy in spagnolo e italiano).

---
## Installazione

Installa solo ciò che manca e carica i modelli di spaCy.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("spacy", "spacy")
asegurar("nltk", "nltk")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")

import nltk
import spacy

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

def cargar_modelo(nombre):
    """Carica un modello di spaCy e lo scarica se non è installato."""
    try:
        return spacy.load(nombre)
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", nombre, "-q"], check=True)
        return spacy.load(nombre)

nlp_es = cargar_modelo("es_core_news_sm")
nlp_it = cargar_modelo("it_core_news_sm")
print("Risorse di NLTK e modelli di spaCy pronti.")

---
## Cella 0 — Setup e dati

In [ ]:
import re
from collections import Counter

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

def tokenizar(texto):
    """Tokenizzazione della lezione 2."""
    return re.findall(r"\w+", texto)

NOMBRE_NLTK = {"spa": "spanish", "ita": "italian"}
SW = {idioma: set(stopwords.words(nombre)) for idioma, nombre in NOMBRE_NLTK.items()}
STEMMER = {idioma: SnowballStemmer(nombre) for idioma, nombre in NOMBRE_NLTK.items()}
NLP = {"spa": nlp_es, "ita": nlp_it}
print("Setup pronto.")

In [ ]:
# datos-es
NEGACIONES = {
    "spa": {"no", "nunca", "jamás", "ni", "sin", "nada", "nadie", "tampoco"},
    "ita": {"non", "mai", "né", "senza", "niente", "nessuno", "nulla", "neanche"},
}
DOMINIO = {"spa": {"pedido", "pedidos"}, "ita": {"ordine", "ordini"}}

TEXTO_EJ2 = {
    "spa": "Los pedidos llegaron tarde. El pedido llegó roto y pedimos otro pedido. Queremos devolver los pedidos rotos.",
    "ita": "Gli ordini sono arrivati tardi. L'ordine è arrivato rotto e abbiamo ordinato un altro ordine. Vogliamo restituire gli ordini rotti.",
}

PARES = {
    "spa": [("universidad", "universo", False), ("generar", "generación", True), ("generoso", "generar", False),
            ("venir", "viene", True), ("ir", "voy", True), ("pedido", "pedidos", True), ("cantar", "cantaron", True),
            ("organizar", "órgano", False), ("entrega", "entregar", True)],
    "ita": [("università", "universo", False), ("generare", "generazione", True), ("generoso", "generare", False),
            ("venire", "viene", True), ("andare", "vado", True), ("ordine", "ordini", True), ("cantare", "cantarono", True),
            ("consegna", "consegnare", True), ("ordine", "ordinare", False)],
}

TEXTO_EJ4 = {
    "spa": "Los clientes cobraron la factura y los paquetes llegaron tarde. Mi hermana vive en Madrid y comieron juntos.",
    "ita": "I clienti hanno pagato la fattura e i pacchi sono arrivati in ritardo. Mia sorella vive a Milano e hanno ordinato la cena.",
}

FRASES_RETO = {
    "spa": ["el pedido no llegó", "no está bien", "los pedidos llegaron rotos y no funcionan", "sin problemas con el envío",
            "nunca fue rápido el envío", "Ayer vino mi hermana y no compró nada"],
    "ita": ["l'ordine non è arrivato", "non è buono", "gli ordini sono arrivati rotti e non funzionano", "senza problemi con la spedizione",
            "la spedizione non è mai stata veloce", "Ieri è venuta mia sorella e non ha comprato niente"],
}

## Esercizio 1 — Un filtro di stopword che conserva le negazioni

Completa `filtrar_stopwords(tokens, idioma, conservar)`: restituisce i token che **non** sono nella lista di NLTK (`SW[idioma]`), confrontando in minuscolo, ma **conservando sempre** quelli presenti nell'insieme `conservar` (anche se sono nella lista). Restituisci i token originali, senza cambiare le maiuscole e rispettando l'ordine. Con `conservar=()` si comporta come il filtro standard.

Prima di programmare, calcola a mano cosa dovrebbe restare di `no está bien` con il filtro standard (ricorda: `no` e `está` sono stopword di NLTK) e con le negazioni conservate.

In [ ]:
def filtrar_stopwords(tokens, idioma, conservar=()):
    resultado = None      # TODO: lista di token: fuori da SW[idioma] o dentro a conservar (confrontando in minuscolo)
    return resultado

# Controllo automatico
assert filtrar_stopwords(["no", "está", "bien"], "spa") is not None, "Completa le righe contrassegnate con TODO"  # dato-es
assert filtrar_stopwords(["no", "está", "bien"], "spa") == ["bien"], "senza conservare nulla, il filtro standard lascia solo «bien»"  # dato-es
assert filtrar_stopwords(["no", "está", "bien"], "spa", NEGACIONES["spa"]) == ["no", "bien"], "con le negazioni conservate restano «no» e «bien»"  # dato-es
assert filtrar_stopwords(["non", "è", "buono"], "ita", NEGACIONES["ita"]) == ["non", "buono"], "in italiano: «non» e «buono»"
assert filtrar_stopwords(["Non", "è", "buono"], "ita", NEGACIONES["ita"]) == ["Non", "buono"], "confronta in minuscolo ma restituisci il token originale"
assert filtrar_stopwords(["sin", "problemas"], "spa") == ["problemas"] and filtrar_stopwords(["senza", "problemi"], "ita") == ["senza", "problemi"], "«sin» è una stopword di NLTK; «senza» no"  # dato-es
assert filtrar_stopwords(tokenizar("nunca fue rápido"), "spa") == ["nunca", "rápido"], "«nunca» non è nella lista di NLTK"  # dato-es
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Quanto si riduce il vocabolario con lo stemming

In `TEXTO_EJ2` hai un testo breve per lingua. Per ciascuno, tokenizza con `tokenizar`, passa al minuscolo e calcola:

- `tipos_spa`, `tipos_ita`: numero di **tipi** (parole distinte) del testo in minuscolo,
- `raices_spa`, `raices_ita`: numero di **radici distinte** dopo aver applicato lo stemmer Snowball di ciascuna lingua,
- `reduccion_spa`, `reduccion_ita`: riduzione percentuale del vocabolario, `100 * (1 - raices / tipos)`, arrotondata a 1 decimale.

Conta a mano i tipi del testo in spagnolo prima di programmare (ci sono 17 token; `pedido` e `pedidos` sono tipi distinti).

In [ ]:
tipos_spa = tipos_ita = None          # TODO: n. di parole distinte in minuscolo
raices_spa = raices_ita = None        # TODO: n. di radici distinte con Snowball
reduccion_spa = reduccion_ita = None  # TODO: 100 * (1 - raices / tipos), 1 decimale

# Controllo automatico
assert all(v is not None for v in (tipos_spa, tipos_ita, raices_spa, raices_ita, reduccion_spa, reduccion_ita)), "Completa le righe contrassegnate con TODO"
assert (tipos_spa, tipos_ita) == (14, 18), "rivedi i tipi: minuscolo e senza togliere stopword"
assert (raices_spa, raices_ita) == (10, 14), "usa STEMMER[idioma].stem(parola) su ogni tipo"
assert (reduccion_spa, reduccion_ita) == (28.6, 22.2), "100 * (1 - raices / tipos) arrotondato a 1 decimale"
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Sovra-stemming e sotto-stemming

`PARES[idioma]` contiene triple `(palabra_1, palabra_2, misma_palabra)`, dove `misma_palabra` indica se le due forme sono della **stessa** parola (`True`) o di parole diverse (`False`). Completa `clasificar_par` in modo che restituisca:

- `"sovra"` se le parole sono **diverse** ma hanno la **stessa radice** (sovra-stemming),
- `"sotto"` se sono della **stessa** parola ma hanno **radici diverse** (sotto-stemming),
- `"ok"` negli altri due casi.

Poi compila `etiquetas` con l'etichetta di ogni coppia, nell'ordine di `PARES`.

In [ ]:
def clasificar_par(palabra_1, palabra_2, misma_palabra, idioma):
    raiz_1 = STEMMER[idioma].stem(palabra_1)
    raiz_2 = STEMMER[idioma].stem(palabra_2)
    etiqueta = None       # TODO: "sovra", "sotto" o "ok" secondo le radici e misma_palabra
    return etiqueta

etiquetas = {idioma: None for idioma in PARES}     # TODO: lista di etichette di ogni lingua, nell'ordine di PARES

# Controllo automatico
assert all(v is not None for v in etiquetas.values()), "Completa le righe contrassegnate con TODO"
assert etiquetas["spa"] == ["sovra", "ok", "sovra", "sotto", "sotto", "ok", "ok", "ok", "ok"], f"spagnolo: rivedi {etiquetas['spa']}"
assert etiquetas["ita"] == ["sovra", "ok", "sovra", "sotto", "sotto", "ok", "ok", "ok", "sovra"], f"italiano: rivedi {etiquetas['ita']}"
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Lemmi di verbi con spaCy

Completa `lemas_verbos(texto, idioma)`: elabora il testo con `NLP[idioma]` e restituisce un dizionario `{forma_in_minuscolo: lemma}` solo con i token il cui `pos_` è `"VERB"`. Poi controlla in `TEXTO_EJ4` i verbi regolari comuni.

> **Nota:** il controllo richiede soltanto l'infinito di pochi verbi regolari, che qualunque versione ragionevole di spaCy indovina. Non controlliamo altro perché i risultati di un modello piccolo possono cambiare con la versione.

In [ ]:
def lemas_verbos(texto, idioma):
    resultado = None      # TODO: dizionario {forma in minuscolo: lemma} dei token con pos_ == "VERB"
    return resultado

verbos_spa = lemas_verbos(TEXTO_EJ4["spa"], "spa")
verbos_ita = lemas_verbos(TEXTO_EJ4["ita"], "ita")

# Controllo automatico
assert verbos_spa is not None and verbos_ita is not None, "Completa le righe contrassegnate con TODO"
for forma, lema in {"cobraron": "cobrar", "llegaron": "llegar", "comieron": "comer", "vive": "vivir"}.items():
    assert verbos_spa.get(forma) == lema, f"il lemma di «{forma}» dovrebbe essere «{lema}»"
for forma, lema in {"pagato": "pagare", "arrivati": "arrivare", "vive": "vivere", "ordinato": "ordinare"}.items():
    assert verbos_ita.get(forma) == lema, f"il lemma di «{forma}» dovrebbe essere «{lema}»"
assert "cliente" not in verbos_spa and "cliente" not in verbos_ita, "solo verbi: i sostantivi restano fuori"
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Scegliere il preprocessing per tre scenari

Per ogni scenario di `ESCENARIOS`, compila in `PLAN` tre campi:

- `"stopwords"`: `"nessuna"` (non si toglie nulla), `"conservare_negazioni"` (lista di NLTK senza le negazioni) o `"standard"` (lista di NLTK completa),
- `"normalizacion"`: `"nessuna"`, `"stemming"` o `"lemmatizzazione"`,
- `"motivo"`: una frase (di almeno 20 caratteri) che giustifichi la scelta.

Non c'è un'unica risposta corretta: il controllo richiede solo i **vincoli** che ogni compito impone (vedi il capitolo, sezione «Guida alla decisione»): un'analisi del sentiment non deve usare la lista standard (si perderebbero le negazioni), e il riconoscimento di entità e la generazione di testo hanno bisogno del testo intatto.

In [ ]:
ESCENARIOS = {
    "sentiment": "Classificare recensioni come positive o negative; molte contengono negazioni («non è buono»).",
    "ner": "Trovare nomi di persone e di aziende nelle email.",
    "generazione": "Addestrare un modello che generi risposte di assistenza in frasi complete.",
}
PLAN = {
    escenario: {"stopwords": None, "normalizacion": None, "motivo": None}     # TODO: compila i tre campi di ogni scenario
    for escenario in ESCENARIOS
}

# Controllo automatico
OPC_SW = {"nessuna", "conservare_negazioni", "standard"}
OPC_NORM = {"nessuna", "stemming", "lemmatizzazione"}
for escenario, decision in PLAN.items():
    assert decision["stopwords"] in OPC_SW and decision["normalizacion"] in OPC_NORM, f"{escenario}: compila stopwords e normalizacion con valori validi"
    assert isinstance(decision["motivo"], str) and len(decision["motivo"]) >= 20, f"{escenario}: giustifica la scelta (almeno 20 caratteri)"
assert PLAN["sentiment"]["stopwords"] != "standard", "la lista standard elimina «no» e «non»: cambia il sentiment"
assert PLAN["ner"]["stopwords"] == "nessuna" and PLAN["ner"]["normalizacion"] == "nessuna", "il riconoscimento di entità ha bisogno del testo intatto"
assert PLAN["generazione"]["stopwords"] == "nessuna" and PLAN["generazione"]["normalizacion"] == "nessuna", "per generare testo non si toglie né si taglia nulla"
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Una funzione `normalizar` con più modalità

Scrivi `normalizar(texto, idioma, modo)` che restituisca una lista di token secondo la modalità:

- `"minimo"`: tokenizza e passa al minuscolo (già fatto),
- `"sin_stop"`: in più, toglie le stopword con una **lista propria**: quella di NLTK **meno** le negazioni di `NEGACIONES[idioma]` **più** le parole di `DOMINIO[idioma]`,
- `"raices"`: come `"sin_stop"` e poi stemming Snowball di ogni token,
- `"lemas"`: lemmi di spaCy (in minuscolo) dei token alfanumerici che non sono nella lista propria (il lemmatizzatore riceve il testo intero, non i token singoli).

Il test controlla due cose su `FRASES_RETO`: che **nessuna modalità elimini mai la negazione** («no» in spagnolo, «non» in italiano) e che il vocabolario di `"raices"` sia minore di quello di `"minimo"`.

> **Nota onesta:** questi controlli si fanno su sei frasi per lingua: servono a vedere che la funzione si comporta bene, **non a sapere quale modalità sia la migliore**. Quella decisione si prende con dati reali e con validazione (a partire dalla lezione 5).

In [ ]:
SW_PERS = {idioma: (SW[idioma] - NEGACIONES[idioma]) | DOMINIO[idioma] for idioma in SW}

def normalizar(texto, idioma, modo="minimo"):
    if modo == "minimo":
        return [t.lower() for t in tokenizar(texto)]
    # TODO 1: modo "sin_stop": minuscolo e fuori le stopword di SW_PERS[idioma]
    # TODO 2: modo "raices": quanto sopra più STEMMER[idioma].stem
    # TODO 3: modo "lemas": lemmi di NLP[idioma](texto), in minuscolo, dei token alfanumerici fuori da SW_PERS
    return None

# Controllo automatico
NEG = {"spa": "no", "ita": "non"}
for idioma, frases in FRASES_RETO.items():
    for modo in ("minimo", "sin_stop", "raices", "lemas"):
        for frase in frases:
            resultado = normalizar(frase, idioma, modo)
            assert resultado is not None, f"la modalità {modo} non è implementata"
            if NEG[idioma] in [t.lower() for t in tokenizar(frase)]:
                assert NEG[idioma] in resultado, f"la modalità {modo} ha eliminato la negazione in: {frase!r}"
    vocab_minimo = {t for f in frases for t in normalizar(f, idioma, "minimo")}
    vocab_raices = {t for f in frases for t in normalizar(f, idioma, "raices")}
    assert len(vocab_raices) < len(vocab_minimo), f"{idioma}: il vocabolario di radici ({len(vocab_raices)}) dovrebbe essere minore del minimo ({len(vocab_minimo)})"
print("Sfida finale corretta ✔")

**Le tue conclusioni (scrivi 3-4 frasi):** quale modalità di `normalizar` sceglieresti per classificare ticket e quale per analizzare il sentiment? Cosa succederebbe alla tua funzione con un testo rumoroso (refusi, allungamenti, clitici)? Come verificheresti, con dati separati da quelli usati per decidere, che la tua scelta è migliore?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def filtrar_stopwords(tokens, idioma, conservar=()):
    resultado = [t for t in tokens if t.lower() not in SW[idioma] or t.lower() in conservar]
    return resultado

assert filtrar_stopwords(["no", "está", "bien"], "spa") is not None, "Completa le righe contrassegnate con TODO"  # dato-es
assert filtrar_stopwords(["no", "está", "bien"], "spa") == ["bien"], "senza conservare nulla, il filtro standard lascia solo «bien»"  # dato-es
assert filtrar_stopwords(["no", "está", "bien"], "spa", NEGACIONES["spa"]) == ["no", "bien"], "con le negazioni conservate restano «no» e «bien»"  # dato-es
assert filtrar_stopwords(["non", "è", "buono"], "ita", NEGACIONES["ita"]) == ["non", "buono"], "in italiano: «non» e «buono»"
assert filtrar_stopwords(["Non", "è", "buono"], "ita", NEGACIONES["ita"]) == ["Non", "buono"], "confronta in minuscolo ma restituisci il token originale"
assert filtrar_stopwords(["sin", "problemas"], "spa") == ["problemas"] and filtrar_stopwords(["senza", "problemi"], "ita") == ["senza", "problemi"], "«sin» è una stopword di NLTK; «senza» no"  # dato-es
assert filtrar_stopwords(tokenizar("nunca fue rápido"), "spa") == ["nunca", "rápido"], "«nunca» non è nella lista di NLTK"  # dato-es

### Soluzione dell'esercizio 2

In [ ]:
def contar(texto, idioma):
    tipos = {t.lower() for t in tokenizar(texto)}
    raices = {STEMMER[idioma].stem(t) for t in tipos}
    return len(tipos), len(raices)

tipos_spa, raices_spa = contar(TEXTO_EJ2["spa"], "spa")
tipos_ita, raices_ita = contar(TEXTO_EJ2["ita"], "ita")
reduccion_spa = round(100 * (1 - raices_spa / tipos_spa), 1)
reduccion_ita = round(100 * (1 - raices_ita / tipos_ita), 1)

assert all(v is not None for v in (tipos_spa, tipos_ita, raices_spa, raices_ita, reduccion_spa, reduccion_ita)), "Completa le righe contrassegnate con TODO"
assert (tipos_spa, tipos_ita) == (14, 18), "rivedi i tipi: minuscolo e senza togliere stopword"
assert (raices_spa, raices_ita) == (10, 14), "usa STEMMER[idioma].stem(parola) su ogni tipo"
assert (reduccion_spa, reduccion_ita) == (28.6, 22.2), "100 * (1 - raices / tipos) arrotondato a 1 decimale"

### Soluzione dell'esercizio 3

In [ ]:
def clasificar_par(palabra_1, palabra_2, misma_palabra, idioma):
    raiz_1 = STEMMER[idioma].stem(palabra_1)
    raiz_2 = STEMMER[idioma].stem(palabra_2)
    if raiz_1 == raiz_2 and not misma_palabra:
        etiqueta = "sovra"
    elif raiz_1 != raiz_2 and misma_palabra:
        etiqueta = "sotto"
    else:
        etiqueta = "ok"
    return etiqueta

etiquetas = {idioma: [clasificar_par(a, b, misma, idioma) for a, b, misma in pares] for idioma, pares in PARES.items()}

assert all(v is not None for v in etiquetas.values()), "Completa le righe contrassegnate con TODO"
assert etiquetas["spa"] == ["sovra", "ok", "sovra", "sotto", "sotto", "ok", "ok", "ok", "ok"], f"spagnolo: rivedi {etiquetas['spa']}"
assert etiquetas["ita"] == ["sovra", "ok", "sovra", "sotto", "sotto", "ok", "ok", "ok", "sovra"], f"italiano: rivedi {etiquetas['ita']}"

### Soluzione dell'esercizio 4

In [ ]:
def lemas_verbos(texto, idioma):
    resultado = {t.text.lower(): t.lemma_ for t in NLP[idioma](texto) if t.pos_ == "VERB"}
    return resultado

verbos_spa = lemas_verbos(TEXTO_EJ4["spa"], "spa")
verbos_ita = lemas_verbos(TEXTO_EJ4["ita"], "ita")

assert verbos_spa is not None and verbos_ita is not None, "Completa le righe contrassegnate con TODO"
for forma, lema in {"cobraron": "cobrar", "llegaron": "llegar", "comieron": "comer", "vive": "vivir"}.items():
    assert verbos_spa.get(forma) == lema, f"il lemma di «{forma}» dovrebbe essere «{lema}»"
for forma, lema in {"pagato": "pagare", "arrivati": "arrivare", "vive": "vivere", "ordinato": "ordinare"}.items():
    assert verbos_ita.get(forma) == lema, f"il lemma di «{forma}» dovrebbe essere «{lema}»"
assert "cliente" not in verbos_spa and "cliente" not in verbos_ita, "solo verbi: i sostantivi restano fuori"

### Soluzione dell'esercizio 5

In [ ]:
ESCENARIOS = {
    "sentiment": "Classificare recensioni come positive o negative; molte contengono negazioni («non è buono»).",
    "ner": "Trovare nomi di persone e di aziende nelle email.",
    "generazione": "Addestrare un modello che generi risposte di assistenza in frasi complete.",
}
PLAN = {
    "sentiment": {"stopwords": "conservare_negazioni", "normalizacion": "lemmatizzazione",
                     "motivo": "Le negazioni invertono il senso; il lemma unisce forme della stessa parola."},
    "ner": {"stopwords": "nessuna", "normalizacion": "nessuna",
            "motivo": "Le maiuscole e il contesto rivelano i nomi propri; non bisogna toccare il testo."},
    "generazione": {"stopwords": "nessuna", "normalizacion": "nessuna",
                  "motivo": "Il testo generato deve essere testo completo e corretto, con tutte le sue parole."},
}

OPC_SW = {"nessuna", "conservare_negazioni", "standard"}
OPC_NORM = {"nessuna", "stemming", "lemmatizzazione"}
for escenario, decision in PLAN.items():
    assert decision["stopwords"] in OPC_SW and decision["normalizacion"] in OPC_NORM, f"{escenario}: compila stopwords e normalizacion con valori validi"
    assert isinstance(decision["motivo"], str) and len(decision["motivo"]) >= 20, f"{escenario}: giustifica la scelta (almeno 20 caratteri)"
assert PLAN["sentiment"]["stopwords"] != "standard", "la lista standard elimina «no» e «non»: cambia il sentiment"
assert PLAN["ner"]["stopwords"] == "nessuna" and PLAN["ner"]["normalizacion"] == "nessuna", "il riconoscimento di entità ha bisogno del testo intatto"
assert PLAN["generazione"]["stopwords"] == "nessuna" and PLAN["generazione"]["normalizacion"] == "nessuna", "per generare testo non si toglie né si taglia nulla"

### Soluzione della sfida finale

In [ ]:
SW_PERS = {idioma: (SW[idioma] - NEGACIONES[idioma]) | DOMINIO[idioma] for idioma in SW}

def normalizar(texto, idioma, modo="minimo"):
    if modo == "minimo":
        return [t.lower() for t in tokenizar(texto)]
    if modo == "lemas":
        return [t.lemma_.lower() for t in NLP[idioma](texto)
                if re.fullmatch(r"\w+", t.text) and t.text.lower() not in SW_PERS[idioma]]
    tokens = [t.lower() for t in tokenizar(texto) if t.lower() not in SW_PERS[idioma]]
    if modo == "raices":
        tokens = [STEMMER[idioma].stem(t) for t in tokens]
    return tokens

NEG = {"spa": "no", "ita": "non"}
for idioma, frases in FRASES_RETO.items():
    for modo in ("minimo", "sin_stop", "raices", "lemas"):
        for frase in frases:
            resultado = normalizar(frase, idioma, modo)
            assert resultado is not None, f"la modalità {modo} non è implementata"
            if NEG[idioma] in [t.lower() for t in tokenizar(frase)]:
                assert NEG[idioma] in resultado, f"la modalità {modo} ha eliminato la negazione in: {frase!r}"
    vocab_minimo = {t for f in frases for t in normalizar(f, idioma, "minimo")}
    vocab_raices = {t for f in frases for t in normalizar(f, idioma, "raices")}
    assert len(vocab_raices) < len(vocab_minimo), f"{idioma}: il vocabolario di radici ({len(vocab_raices)}) dovrebbe essere minore del minimo ({len(vocab_minimo)})"
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto non ti è rimasto chiaro (stopword e negazioni, sovra-stemming, sotto-stemming, POS e lemma, cosa preprocessare secondo il compito)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*